# Week 13 — Applied Example: Poisson Regression on Count Data
## Modelos de Análisis Estadístico | Universidad de los Andes
### Prof. Alejandra Tabares

---

## Context

This notebook presents a **full applied pipeline** for count data modelling, inspired by the classic *Biochemists* dataset (Long & Freese) but simulated here so the notebook is entirely self-contained.

**Research question:** What factors predict the number of articles published by PhD biochemists during the last three years of their doctoral program?

**Outcome variable:** `articles` — number of publications (non-negative integer).

**Predictors:**

| Variable | Description |
|----------|-------------|
| `female` | Gender (1 = female, 0 = male) |
| `married` | Marital status (1 = married, 0 = not married) |
| `children` | Number of children aged 5 or younger |
| `prestige` | Prestige score of PhD program (continuous, standardised) |
| `mentor_articles` | Number of articles published by the PhD mentor |

**Pipeline:**
1. Data simulation and description
2. Exploratory Data Analysis (EDA)
3. Poisson regression: fit and interpret
4. Residual diagnostics
5. Test for overdispersion
6. Negative Binomial regression as alternative
7. Model comparison
8. Conclusions

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

np.random.seed(2026)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

print('Environment ready.')

---

## Step 1: Data Simulation

We simulate 915 PhD students (matching the original Biochemists dataset size) from the following data-generating process:

$$\log(\mu_i) = 0.30 - 0.20 \cdot \text{female}_i + 0.10 \cdot \text{married}_i - 0.15 \cdot \text{children}_i + 0.15 \cdot \text{prestige}_i + 0.025 \cdot \text{mentor\_articles}_i$$

Counts are drawn from a **Negative Binomial** distribution (dispersion $\alpha = 0.5$) to introduce realistic overdispersion.

In [ ]:
n = 915

# ── Predictors ───────────────────────────────────────────────────────────────
female          = np.random.binomial(1, 0.46, n)
married         = np.random.binomial(1, 0.66, n)
children        = np.random.poisson(0.5, n)           # mostly 0, 1, 2
prestige        = np.random.normal(0, 1, n)            # standardised score
mentor_articles = np.random.negative_binomial(3, 0.3, n)  # right-skewed

# ── True log-mean ─────────────────────────────────────────────────────────
log_mu = (0.30
          - 0.20 * female
          + 0.10 * married
          - 0.15 * children
          + 0.15 * prestige
          + 0.025 * mentor_articles)
mu_true = np.exp(log_mu)

# ── Overdispersed counts (Negative Binomial with α = 0.5) ────────────────
alpha_true = 0.5
r_nb = 1.0 / alpha_true
p_nb = r_nb / (r_nb + mu_true)
articles = np.random.negative_binomial(r_nb, p_nb)

# ── Assemble DataFrame ───────────────────────────────────────────────────
df = pd.DataFrame({
    'articles':        articles,
    'female':          female,
    'married':         married,
    'children':        children,
    'prestige':        prestige,
    'mentor_articles': mentor_articles
})

print(f'Dataset: {df.shape[0]} observations, {df.shape[1]} variables')
print()
print(df.head(10))

---

## Step 2: Exploratory Data Analysis (EDA)

Before fitting any model we examine:
- The distribution of the outcome (`articles`).
- Whether mean ≈ variance (Poisson equidispersion).
- Relationships between predictors and the outcome.

In [ ]:
# ── Descriptive statistics ────────────────────────────────────────────────
desc = df.describe().T
print(desc.round(3).to_string())

In [ ]:
# ── Count frequency table for the outcome ────────────────────────────────
freq = df['articles'].value_counts().sort_index()
print('Frequency table — articles published:')
print(pd.DataFrame({'count': freq, 'pct': (freq / len(df) * 100).round(1)}).head(15))

mean_art = df['articles'].mean()
var_art  = df['articles'].var()
print(f'\nMean     = {mean_art:.3f}')
print(f'Variance = {var_art:.3f}')
print(f'Variance / Mean = {var_art / mean_art:.3f}  (> 1 suggests overdispersion)')

In [ ]:
# ── Figure 1: Distribution of articles + Poisson overlay ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Observed count distribution
max_k = int(df['articles'].quantile(0.99))
obs_freq = df['articles'].value_counts().reindex(range(max_k + 1), fill_value=0) / len(df)

x_vals = np.arange(max_k + 1)
poisson_pmf  = stats.poisson.pmf(x_vals, mean_art)

axes[0].bar(x_vals - 0.2, obs_freq.values, width=0.35,
            label='Observed', color='steelblue', alpha=0.8)
axes[0].bar(x_vals + 0.2, poisson_pmf, width=0.35,
            label=f'Poisson(λ={mean_art:.2f})', color='coral', alpha=0.8)
axes[0].set(xlabel='Number of articles', ylabel='Proportion',
            title='Observed vs Poisson Expected')
axes[0].legend()
axes[0].set_xlim(-0.5, max_k + 0.5)

# Mean vs Variance
axes[1].bar(['Mean', 'Variance'], [mean_art, var_art],
            color=['steelblue', 'coral'], alpha=0.85, edgecolor='white')
for i, val in enumerate([mean_art, var_art]):
    axes[1].text(i, val + 0.15, f'{val:.2f}', ha='center', fontsize=11)
axes[1].set(title='Mean vs Variance of Articles', ylabel='Value')

plt.suptitle('EDA — Distribution of Publication Counts', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 2: Outcome vs predictors ──────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

# Box plots for binary predictors
for ax, var, label in zip(
    [axes[0, 0], axes[0, 1], axes[0, 2]],
    ['female', 'married', 'children'],
    ['Gender (0=male, 1=female)', 'Married', 'Children (≤5 yrs)']
):
    df.boxplot(column='articles', by=var, ax=ax,
               boxprops=dict(color='steelblue'),
               medianprops=dict(color='coral', linewidth=2),
               flierprops=dict(marker='o', markersize=3, alpha=0.3))
    ax.set(xlabel=label, ylabel='Articles', title='')
    plt.sca(ax)
    plt.title(f'Articles by {label}')

# Scatter for continuous predictors
for ax, var, label in zip(
    [axes[1, 0], axes[1, 1], axes[1, 2]],
    ['prestige', 'mentor_articles', 'prestige'],  # prestige twice for variety
    ['Program Prestige', 'Mentor Articles', 'Prestige vs Log(articles+1)']
):
    if label == 'Prestige vs Log(articles+1)':
        ax.scatter(df[var], np.log1p(df['articles']),
                   alpha=0.25, s=12, color='steelblue')
        ax.set(xlabel=label.split(' vs ')[0],
               ylabel='log(articles + 1)', title=label)
        # Regression line
        m, b = np.polyfit(df[var], np.log1p(df['articles']), 1)
        xs = np.linspace(df[var].min(), df[var].max(), 100)
        ax.plot(xs, m * xs + b, 'r-', linewidth=1.5)
    else:
        ax.scatter(df[var], df['articles'], alpha=0.25, s=12, color='steelblue')
        ax.set(xlabel=label, ylabel='Articles', title=f'Articles vs {label}')

plt.suptitle('EDA — Outcome vs Predictors', fontsize=13)
plt.tight_layout()
plt.show()

---

## Step 3: Fit Poisson Regression

In [ ]:
# Fit Poisson GLM
poisson_fit = smf.glm(
    formula='articles ~ female + married + children + prestige + mentor_articles',
    data=df,
    family=sm.families.Poisson()
).fit()

print(poisson_fit.summary())

In [ ]:
# Rate Ratios table
irr = np.exp(poisson_fit.params)
ci  = np.exp(poisson_fit.conf_int())

irr_df = pd.DataFrame({
    'β (log scale)':  poisson_fit.params,
    'IRR':            irr,
    'CI 2.5%':        ci[0],
    'CI 97.5%':       ci[1],
    'p-value':        poisson_fit.pvalues
}).round(4)

print('Poisson Regression — Rate Ratios:')
print(irr_df)

### Interpretation of Coefficients

Reading the Incidence Rate Ratios (IRR) from the table above:

- **`female`**: Controlling for other variables, female students publish at a rate approximately $e^{\hat{\beta}_{\text{female}}}$ times that of male students. An IRR below 1 indicates fewer publications on average.
- **`married`**: Married students have a publication rate $e^{\hat{\beta}_{\text{married}}}$ times that of unmarried students.
- **`children`**: Each additional child is associated with a multiplicative change of $e^{\hat{\beta}_{\text{children}}}$ in the expected number of publications.
- **`prestige`**: A one-standard-deviation increase in program prestige multiplies the expected count by $e^{\hat{\beta}_{\text{prestige}}}$.
- **`mentor_articles`**: Each additional article published by the mentor is associated with a factor change of $e^{\hat{\beta}_{\text{mentor}}}$ in the student's expected output.

In [ ]:
# Forest plot of IRRs
irr_plot = irr_df.drop('Intercept')

fig, ax = plt.subplots(figsize=(8, 4))
y_pos = range(len(irr_plot))

colors = ['coral' if v < 1 else 'steelblue' for v in irr_plot['IRR']]
ax.scatter(irr_plot['IRR'], list(y_pos), color=colors, zorder=3, s=100)
for i, (idx, row) in enumerate(irr_plot.iterrows()):
    ax.hlines(i, row['CI 2.5%'], row['CI 97.5%'],
              color=colors[i], linewidth=2.5, alpha=0.7)

ax.axvline(1, color='black', linestyle='--', linewidth=1.2, label='IRR = 1')
ax.set_yticks(list(y_pos))
ax.set_yticklabels(irr_plot.index)
ax.set(xlabel='Incidence Rate Ratio (IRR)', xlim=(0.5, 2.0),
       title='Poisson Regression — Rate Ratios with 95% CI')
ax.legend()
plt.tight_layout()
plt.show()

---

## Step 4: Residual Diagnostics

We examine three types of residuals:

1. **Pearson residuals**: $(y_i - \hat{\mu}_i) / \sqrt{\hat{\mu}_i}$ — should have mean ≈ 0 and variance ≈ 1 under correct specification.
2. **Deviance residuals**: sign$(y_i - \hat{\mu}_i) \sqrt{d_i}$ — contribution of each observation to the residual deviance.
3. **Predicted vs Observed** scatter — patterns indicate model misspecification.

In [ ]:
mu_hat    = poisson_fit.predict()
pearson_r = poisson_fit.resid_pearson
deviance_r = poisson_fit.resid_deviance

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# 1. Pearson residuals vs fitted
axes[0, 0].scatter(mu_hat, pearson_r, alpha=0.3, s=12, color='steelblue')
axes[0, 0].axhline(0, color='red', linestyle='--')
axes[0, 0].axhline( 2, color='orange', linestyle=':', linewidth=1)
axes[0, 0].axhline(-2, color='orange', linestyle=':', linewidth=1)
axes[0, 0].set(xlabel='Fitted values (μ̂)', ylabel='Pearson residuals',
               title='Pearson Residuals vs Fitted')

# 2. Deviance residuals vs fitted
axes[0, 1].scatter(mu_hat, deviance_r, alpha=0.3, s=12, color='darkorange')
axes[0, 1].axhline(0, color='red', linestyle='--')
axes[0, 1].axhline( 2, color='orange', linestyle=':', linewidth=1)
axes[0, 1].axhline(-2, color='orange', linestyle=':', linewidth=1)
axes[0, 1].set(xlabel='Fitted values (μ̂)', ylabel='Deviance residuals',
               title='Deviance Residuals vs Fitted')

# 3. Histogram of Pearson residuals
axes[1, 0].hist(pearson_r, bins=40, edgecolor='white', color='steelblue')
axes[1, 0].set(xlabel='Pearson residuals', ylabel='Frequency',
               title='Distribution of Pearson Residuals')

# 4. Predicted vs Observed
axes[1, 1].scatter(mu_hat, df['articles'], alpha=0.25, s=12, color='steelblue')
max_v = max(mu_hat.max(), df['articles'].max())
axes[1, 1].plot([0, max_v], [0, max_v], 'r--', label='y = x (perfect fit)')
axes[1, 1].set(xlabel='Predicted μ̂', ylabel='Observed y',
               title='Predicted vs Observed')
axes[1, 1].legend()

plt.suptitle('Poisson Model — Residual Diagnostics', fontsize=13)
plt.tight_layout()
plt.show()

---

## Step 5: Testing for Overdispersion

We use two approaches:

1. **Pearson dispersion statistic** $\hat{\phi} = \chi^2_P / df$.
2. **Cameron & Trivedi (1990) regression test**: regress $(y_i - \hat{\mu}_i)^2 - y_i$ on $\hat{\mu}_i^2$. A significantly positive slope indicates overdispersion of the NB2 form.

In [ ]:
# ── Pearson dispersion statistic ──────────────────────────────────────────
pearson_chi2 = np.sum(pearson_r**2)
df_resid     = poisson_fit.df_resid
phi_hat      = pearson_chi2 / df_resid

print('=== Overdispersion Diagnostics ===')
print(f'Pearson chi² = {pearson_chi2:.2f}')
print(f'Residual df  = {df_resid}')
print(f'φ̂  (Pearson chi² / df) = {phi_hat:.3f}')
print()
if phi_hat > 1.5:
    print('>> Strong overdispersion detected (φ̂ >> 1).')
    print('   Negative Binomial regression is recommended.')
elif phi_hat > 1.1:
    print('>> Mild overdispersion. Consider NB or quasi-Poisson.')
else:
    print('>> No strong overdispersion. Poisson model may be adequate.')

In [ ]:
# ── Cameron & Trivedi regression test ────────────────────────────────────
# H₀: α = 0 (equidispersion)   H₁: α > 0 (overdispersion)
y     = df['articles'].values
mu_p  = poisson_fit.predict().values

z = (y - mu_p)**2 - y          # dependent variable
w = mu_p**2                    # independent variable (NB2 form)

# OLS regression of z on w (no intercept)
w_const = sm.add_constant(w)
ct_ols  = sm.OLS(z, w_const).fit()

alpha_ct = ct_ols.params[1]
pval_ct  = ct_ols.pvalues[1]

print('Cameron & Trivedi (1990) Test for Overdispersion')
print(f'H₀: α = 0  (equidispersion)')
print(f'Estimated α = {alpha_ct:.4f}')
print(f'p-value     = {pval_ct:.4e}')
if pval_ct < 0.05:
    print('>> Reject H₀: significant overdispersion (NB2 form).')
else:
    print('>> Fail to reject H₀: no evidence of overdispersion.')

---

## Step 6: Negative Binomial Regression

Given the evidence of overdispersion, we fit a **Negative Binomial (NB2)** model. The specification is identical to Poisson regression — same predictors, same log link — with the addition of a dispersion parameter $\alpha$ estimated from the data.

In [ ]:
nb_fit = smf.negativebinomial(
    'articles ~ female + married + children + prestige + mentor_articles',
    data=df
).fit(disp=False)

print(nb_fit.summary())

In [ ]:
# Rate Ratios — Negative Binomial
nb_params = nb_fit.params.drop('alpha', errors='ignore')
nb_ci     = nb_fit.conf_int().drop('alpha', errors='ignore')

irr_nb = pd.DataFrame({
    'β (log scale)': nb_params,
    'IRR':           np.exp(nb_params),
    'CI 2.5%':       np.exp(nb_ci[0]),
    'CI 97.5%':      np.exp(nb_ci[1]),
    'p-value':       nb_fit.pvalues.drop('alpha', errors='ignore')
}).round(4)

print('Negative Binomial — Rate Ratios:')
print(irr_nb)
print(f'\nEstimated dispersion α = {nb_fit.params["alpha"]:.4f}  (true α = {alpha_true})')

---

## Step 7: Model Comparison

In [ ]:
# ── AIC / BIC / Log-likelihood ────────────────────────────────────────────
comparison = pd.DataFrame({
    'Model': ['Poisson', 'Negative Binomial'],
    'Log-Likelihood': [poisson_fit.llf, nb_fit.llf],
    'AIC':            [poisson_fit.aic, nb_fit.aic],
    'BIC':            [poisson_fit.bic, nb_fit.bic],
    'Pearson φ':      [phi_hat, np.sum(((df['articles'] - nb_fit.predict())**2
                                        / nb_fit.predict()) / nb_fit.df_resid)]
}).set_index('Model')

print('Model Comparison:')
print(comparison.round(2))
print()
delta_aic = poisson_fit.aic - nb_fit.aic
print(f'ΔAIC (Poisson − NB) = {delta_aic:.2f}')
if delta_aic > 10:
    print('Strong evidence in favour of the Negative Binomial model.')

In [ ]:
# ── Side-by-side coefficient comparison ──────────────────────────────────
coef_compare = pd.DataFrame({
    'Poisson β':  poisson_fit.params,
    'Poisson SE': poisson_fit.bse,
    'NB β':       nb_params.reindex(poisson_fit.params.index),
    'NB SE':      nb_fit.bse.drop('alpha', errors='ignore').reindex(poisson_fit.params.index),
})

print('Coefficient comparison (note inflated Poisson SEs due to overdispersion):')
print(coef_compare.round(4))

In [ ]:
# ── Figure: Predicted vs Observed for both models ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, model, title, color in zip(
    axes,
    [poisson_fit, nb_fit],
    ['Poisson', 'Negative Binomial'],
    ['steelblue', 'darkorange']
):
    pred = model.predict()
    obs  = df['articles'].values
    ax.scatter(pred, obs, alpha=0.25, s=14, color=color)
    max_v = max(pred.max(), obs.max())
    ax.plot([0, max_v], [0, max_v], 'r--', linewidth=1.5, label='y = x')
    corr = np.corrcoef(pred, obs)[0, 1]
    ax.set(xlabel='Predicted μ̂', ylabel='Observed y',
           title=f'{title} — Predicted vs Observed\n(r = {corr:.3f})')
    ax.legend()

plt.suptitle('Predicted vs Observed: Poisson vs Negative Binomial', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure: Observed vs predicted count frequencies ───────────────────────
max_k = int(df['articles'].quantile(0.975))
x_vals = np.arange(max_k + 1)

obs_prop = df['articles'].value_counts().reindex(x_vals, fill_value=0).values / len(df)

# Simulate predicted proportions from both models (parametric bootstrap)
n_sim = 10_000

def predicted_pmf_poisson(model, df, x_vals, n_sim=5000):
    mu_p = model.predict().values
    sims = np.random.poisson(mu_p[np.random.choice(len(mu_p), n_sim)], size=n_sim)
    return np.array([np.mean(sims == k) for k in x_vals])

def predicted_pmf_nb(model, df, x_vals, alpha_est, n_sim=5000):
    mu_p = model.predict().values
    mu_s = mu_p[np.random.choice(len(mu_p), n_sim)]
    r    = 1.0 / alpha_est
    p_s  = r / (r + mu_s)
    sims = np.random.negative_binomial(r, p_s)
    return np.array([np.mean(sims == k) for k in x_vals])

np.random.seed(0)
pois_pmf = predicted_pmf_poisson(poisson_fit, df, x_vals, n_sim)
nb_pmf   = predicted_pmf_nb(nb_fit, df, x_vals, nb_fit.params['alpha'], n_sim)

fig, ax = plt.subplots(figsize=(11, 5))
width = 0.28
ax.bar(x_vals - width, obs_prop,  width, label='Observed',          color='steelblue', alpha=0.85)
ax.bar(x_vals,          pois_pmf, width, label='Poisson predicted',  color='coral',     alpha=0.85)
ax.bar(x_vals + width,  nb_pmf,   width, label='NB predicted',       color='seagreen',  alpha=0.85)
ax.set(xlabel='Number of articles', ylabel='Proportion',
       title='Observed vs Predicted Count Frequencies',
       xlim=(-0.5, max_k + 0.5))
ax.legend()
plt.tight_layout()
plt.show()

---

## Step 8: Conclusions

### Findings from the Poisson Model

The Poisson regression identifies several statistically significant predictors of doctoral publication output. The direction of all estimated rate ratios is consistent with the data-generating process used in the simulation:

- **Gender** (`female`): Female students publish at a lower rate than male students (IRR < 1), holding other factors constant. This difference is statistically significant.
- **Marital status** (`married`): Married students show a slightly higher publication rate (IRR > 1), although the effect is modest.
- **Young children** (`children`): Having more children under age 5 is negatively associated with publication output (IRR < 1). Each additional child multiplies the expected count by a factor below 1.
- **Program prestige** (`prestige`): Higher-prestige programs are associated with higher publication rates. The IRR above 1 reflects the intellectual resources and networks available in top programs.
- **Mentor productivity** (`mentor_articles`): A highly productive mentor is strongly associated with student output. Each additional article by the mentor increases the student's expected count by a factor of $e^{\hat{\beta}_{\text{mentor}}}$, suggesting mentorship or shared environment effects.

### Overdispersion

Both the Pearson dispersion statistic ($\hat{\phi} \gg 1$) and the Cameron & Trivedi test (significant at the 5% level) indicate **strong overdispersion**. The Poisson model underestimates the variance in publication counts, leading to artificially small standard errors and inflated test statistics. This is a common feature of count data in the social sciences.

### Negative Binomial as the Preferred Model

The Negative Binomial model accounts for overdispersion by estimating the dispersion parameter $\alpha > 0$. Compared to the Poisson model:

- The **AIC is substantially lower** for the NB model (ΔAIC >> 10), confirming it as a much better fit.
- The **standard errors are larger** under NB, providing more honest uncertainty estimates.
- The **point estimates** are similar across both models (as expected), because overdispersion affects precision, not consistency of the MLE.
- The **predicted count frequencies** from the NB model track the observed distribution more closely, particularly in the tails.

### Practical Recommendations

1. Always check for overdispersion when modelling count outcomes.
2. If $\hat{\phi} > 1.5$ or the Cameron & Trivedi test is significant, prefer the Negative Binomial model.
3. Report **rate ratios** (exponentiated coefficients) with confidence intervals for clear communication.
4. Use **AIC/BIC** for model comparison between Poisson and NB (they are not nested in the usual sense, but the NB reduces to Poisson when $\alpha = 0$).

---

*Modelos de Análisis Estadístico — Universidad de los Andes, 2026-01*